In [2]:
import requests
import pandas as pd
import time
import os

# ============ الإعدادات ============
# صندوق إحداثيات مصر (Bounding Box) تقريبًا
LAT_MIN, LAT_MAX = 22.0, 31.7
LON_MIN, LON_MAX = 25.0, 35.0

STEP = 0.5   # نفس دقة NASA POWER الأصلية (0.5 درجة)

PARAMETERS = "ALLSKY_SFC_SW_DWN,T2M,RH2M,WS2M"
COMMUNITY = "RE"   # Renewable Energy community
BASE_URL = "https://power.larc.nasa.gov/api/temporal/climatology/point"

OUTPUT_DIR = "nasa_power_egypt_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 50     # كل كام نقطة نعمل حفظ في ملف
SLEEP_TIME = 0.3    # وقفة صغيرة بين كل Request عشان محدش يعمل block للـ IP

In [3]:
def build_grid(lat_min, lat_max, lon_min, lon_max, step):
    lats = [round(lat_min + i * step, 3) for i in range(int((lat_max - lat_min) / step) + 1)]
    lons = [round(lon_min + i * step, 3) for i in range(int((lon_max - lon_min) / step) + 1)]
    grid = [(lat, lon) for lat in lats for lon in lons]
    return grid

grid_points = build_grid(LAT_MIN, LAT_MAX, LON_MIN, LON_MAX, STEP)
print(f"عدد نقاط الشبكة: {len(grid_points)}")

عدد نقاط الشبكة: 420


In [4]:
import geopandas as gpd
from shapely.geometry import Point

# ملف GeoJSON فيه حدود كل دول العالم (مصدر عام موثوق - Natural Earth)
world_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_0_countries.geojson"

world = gpd.read_file(world_url)

# فلترة مصر بس (ممكن يكون الاسم NAME أو ADMIN حسب نسخة الملف)
egypt = world[world["ADMIN"] == "Egypt"]

if egypt.empty:
    egypt = world[world["NAME"] == "Egypt"]

egypt_geom = egypt.geometry.unary_union
print("تم تحميل حدود مصر بنجاح.")

def filter_grid_by_country(grid_points, country_geom):
    filtered = []
    for lat, lon in grid_points:
        point = Point(lon, lat)  # لاحظ: Point(x=lon, y=lat)
        if country_geom.contains(point):
            filtered.append((lat, lon))
    return filtered

grid_points_filtered = filter_grid_by_country(grid_points, egypt_geom)

print(f"عدد النقاط الأصلية (الصندوق كامل): {len(grid_points)}")
print(f"عدد النقاط بعد الفلترة (داخل حدود مصر فعليًا): {len(grid_points_filtered)}")

تم تحميل حدود مصر بنجاح.
عدد النقاط الأصلية (الصندوق كامل): 420
عدد النقاط بعد الفلترة (داخل حدود مصر فعليًا): 377


C:\Users\ZBook\AppData\Local\Temp\ipykernel_9672\3170372193.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  egypt_geom = egypt.geometry.unary_union


In [5]:
def fetch_point_climatology(lat, lon, max_retries=3):
    params = {
        "parameters": PARAMETERS,
        "community": COMMUNITY,
        "longitude": lon,
        "latitude": lat,
        "format": "JSON"
    }

    for attempt in range(max_retries):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()

            props = data["properties"]["parameter"]
            row = {"latitude": lat, "longitude": lon}

            for feature, months in props.items():
                for month, value in months.items():
                    row[f"{feature}_{month}"] = value

            return row

        except Exception as e:
            print(f"⚠️ فشل عند ({lat}, {lon}) - محاولة {attempt+1}: {e}")
            time.sleep(2)

    return None  # فشلت كل المحاولات

In [6]:
results = []
chunk_num = 0
failed_points = []

for i, (lat, lon) in enumerate(grid_points_filtered, start=1):
    row = fetch_point_climatology(lat, lon)
    if row:
        results.append(row)
    else:
        failed_points.append((lat, lon))

    time.sleep(SLEEP_TIME)

    if i % 10 == 0 or i == len(grid_points_filtered):
        print(f"تم {i}/{len(grid_points_filtered)} نقطة...")

    # حفظ كل CHUNK_SIZE نقطة في ملف منفصل
    if len(results) >= CHUNK_SIZE or i == len(grid_points_filtered):
        if results:
            df_chunk = pd.DataFrame(results)
            chunk_num += 1
            out_path = os.path.join(OUTPUT_DIR, f"egypt_climatology_part{chunk_num}.csv")
            df_chunk.to_csv(out_path, index=False)
            print(f"✅ اتحفظ: {out_path} ({len(results)} صف)")
            results = []

print("انتهى السحب.")
if failed_points:
    print(f"عدد النقاط اللي فشلت: {len(failed_points)}")

تم 10/377 نقطة...
تم 20/377 نقطة...
تم 30/377 نقطة...
تم 40/377 نقطة...
تم 50/377 نقطة...
✅ اتحفظ: nasa_power_egypt_data\egypt_climatology_part1.csv (50 صف)
تم 60/377 نقطة...
تم 70/377 نقطة...
تم 80/377 نقطة...
تم 90/377 نقطة...
تم 100/377 نقطة...
✅ اتحفظ: nasa_power_egypt_data\egypt_climatology_part2.csv (50 صف)
تم 110/377 نقطة...
تم 120/377 نقطة...
تم 130/377 نقطة...
تم 140/377 نقطة...
تم 150/377 نقطة...
✅ اتحفظ: nasa_power_egypt_data\egypt_climatology_part3.csv (50 صف)
تم 160/377 نقطة...
تم 170/377 نقطة...
تم 180/377 نقطة...
تم 190/377 نقطة...
تم 200/377 نقطة...
✅ اتحفظ: nasa_power_egypt_data\egypt_climatology_part4.csv (50 صف)
تم 210/377 نقطة...
تم 220/377 نقطة...
تم 230/377 نقطة...
تم 240/377 نقطة...
تم 250/377 نقطة...
✅ اتحفظ: nasa_power_egypt_data\egypt_climatology_part5.csv (50 صف)
تم 260/377 نقطة...
تم 270/377 نقطة...
تم 280/377 نقطة...
تم 290/377 نقطة...
تم 300/377 نقطة...
✅ اتحفظ: nasa_power_egypt_data\egypt_climatology_part6.csv (50 صف)
تم 310/377 نقطة...
تم 320/377 نقطة...

In [7]:
import glob

all_files = glob.glob(os.path.join(OUTPUT_DIR, "egypt_climatology_part*.csv"))
df_all = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
df_all.to_csv(os.path.join(OUTPUT_DIR, "egypt_climatology_full.csv"), index=False)
print(f"الملف النهائي فيه {len(df_all)} صف و {df_all.shape[1]} عمود")
df_all.head()

الملف النهائي فيه 377 صف و 54 عمود


,latitude,longitude,ALLSKY_SFC_SW_DWN_JAN,ALLSKY_SFC_SW_DWN_FEB,ALLSKY_SFC_SW_DWN_MAR,ALLSKY_SFC_SW_DWN_APR,ALLSKY_SFC_SW_DWN_MAY,ALLSKY_SFC_SW_DWN_JUN,ALLSKY_SFC_SW_DWN_JUL,ALLSKY_SFC_SW_DWN_AUG,...,WS2M_APR,WS2M_MAY,WS2M_JUN,WS2M_JUL,WS2M_AUG,WS2M_SEP,WS2M_OCT,WS2M_NOV,WS2M_DEC,WS2M_ANN
0,22.0,25.0,5.1413,5.9870,6.9679,7.6558,7.9409,8.1559,8.1149,7.7208,...,2.94,2.98,3.13,3.00,2.96,3.14,2.86,2.42,2.51,2.85
1,22.0,25.5,5.1413,5.9870,6.9679,7.6558,7.9409,8.1559,8.1149,7.7208,...,2.89,2.93,3.06,2.87,2.84,3.09,2.88,2.48,2.59,2.82
2,22.0,26.0,5.1223,5.9647,6.8962,7.5991,7.9027,8.1403,8.1060,7.7124,...,2.75,2.81,2.97,2.71,2.70,3.09,2.95,2.48,2.50,2.73
3,22.0,26.5,5.1223,5.9647,6.8962,7.5991,7.9027,8.1403,8.1060,7.7124,...,2.75,2.81,2.97,2.71,2.70,3.09,2.95,2.48,2.50,2.73
4,22.0,27.0,5.0724,5.9251,6.8633,7.5895,7.8708,8.1022,8.0494,7.6366,...,2.86,2.95,3.14,2.81,2.85,3.39,3.28,2.73,2.67,2.89
